In [ ]:
import pandas as pd
import numpy as np

from os.path import join
from config import BASE_DIR, DATA_DIR

### Hatespeech labels

In [27]:
hs = pd.read_parquet(join(BASE_DIR, "hate_speech", "openai_output", "hs_per_toot.parquet"))

In [31]:
hs.drop_duplicates(subset=["custom_id", "is_hate"]).shape

(12822517, 3)

In [32]:
hs.drop_duplicates(inplace=True)

In [34]:
hs["instance"] = hs.custom_id.map(lambda x: x.split("_")[0])
hs.instance.nunique()

55475

In [ ]:
hs.instance.value_counts()

### Toots

In [37]:
toots = pd.read_parquet(join(DATA_DIR, "toots", "toots_metadata.parquet"))

In [ ]:
toots.shape

In [40]:
toots = toots[toots.custom_id.isin(hs.custom_id)]

In [42]:
toots.drop_duplicates(subset=["custom_id"], inplace=True)

In [ ]:
toots.shape

In [46]:
hs = pd.merge(hs, toots, on="custom_id", how="left")

In [ ]:
hs.shape

In [50]:
hs["instance"] = hs.custom_id.map(lambda x: x.split("_")[0] if len(x.split("_")) > 1 else None)

In [51]:
hs = hs[~hs.instance.isna()]

## Hate concentration analysis

In [ ]:
account_level = (
    hs.groupby(['instance', 'account'])['is_hate']
      .max()
      .reset_index(name='is_hate_account')
)

instance_summary = (
    account_level.groupby('instance')
    .agg(
        total_accounts=('account', 'nunique'),
        hate_accounts=('is_hate_account', 'sum')
    )
    .reset_index()
)

instance_summary['pct_hate_accounts'] = (
    100 * instance_summary['hate_accounts'] / instance_summary['total_accounts']
)

k = max(1, int(np.ceil(0.10 * len(instance_summary))))
top10_instances = instance_summary.nlargest(k, 'hate_accounts')

top10_share = (
    top10_instances['hate_accounts'].sum() /
    instance_summary['hate_accounts'].sum()
    if instance_summary['hate_accounts'].sum() > 0 else 0.0
)

hate_counts = (
    hs[hs['is_hate']]
    .groupby(['instance', 'account'])
    .size()
    .reset_index(name='hate_count')
)

all_accounts = account_level[['instance', 'account']]

hate_per_account = (
    all_accounts.merge(hate_counts, on=['instance', 'account'], how='left')
    .fillna({'hate_count': 0})
)

hate_per_account['hate_count'] = hate_per_account['hate_count'].astype(int)

def top_share(x, frac=0.01):
    x = np.sort(np.asarray(x))[::-1]
    total = x.sum()
    if total == 0:
        return 0.0
    k = max(1, int(np.ceil(frac * len(x))))
    return x[:k].sum() / total

top1_per_instance = (
    hate_per_account.groupby('instance')['hate_count']
    .apply(lambda x: top_share(x, 0.01))
    .reset_index(name='top1_share')
)

instances_with_hate = instance_summary.loc[instance_summary['hate_accounts'] > 0, 'instance']

top1_per_instance_hate_only = top1_per_instance[
    top1_per_instance['instance'].isin(instances_with_hate)
].copy()

results = {
    # Across all instances
    'median_pct_hate_accounts_per_instance': instance_summary['pct_hate_accounts'].median(),
    'mean_pct_hate_accounts_per_instance': instance_summary['pct_hate_accounts'].mean(),
    'top10_share_of_all_hate_accounts_across_instances': top10_share,

    # Within instances that actually contain hate
    'median_top1_share_within_hate_instances': top1_per_instance_hate_only['top1_share'].median()
        if len(top1_per_instance_hate_only) > 0 else np.nan,
    'iqr_top1_share_within_hate_instances': (
        top1_per_instance_hate_only['top1_share'].quantile(0.25),
        top1_per_instance_hate_only['top1_share'].quantile(0.75)
    ) if len(top1_per_instance_hate_only) > 0 else (np.nan, np.nan),
}

for k, v in results.items():
    print(f"{k}: {v}")


In [ ]:
top1_per_instance[top1_per_instance['top1_share'] > 0]['top1_share'].median()